In [12]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

In [13]:
BASE_DIR = Path.cwd()  # Current working directory

if not (BASE_DIR / "data").exists():
    BASE_DIR =BASE_DIR.parent
training_data_path = BASE_DIR / "data" / "train_data.csv"
test_data_path = BASE_DIR / "data" / "test_data.csv"

train_data = pd.read_csv(training_data_path)
test_data = pd.read_csv(test_data_path)




In [14]:
train_data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,0,No,No,35,No,No phone service,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,0
1,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55,0
2,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,0
3,Female,0,Yes,No,26,Yes,No,DSL,No,Yes,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.70,0
4,Male,0,Yes,Yes,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55,0


In [15]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            5634 non-null   str    
 1   SeniorCitizen     5634 non-null   int64  
 2   Partner           5634 non-null   str    
 3   Dependents        5634 non-null   str    
 4   tenure            5634 non-null   int64  
 5   PhoneService      5634 non-null   str    
 6   MultipleLines     5634 non-null   str    
 7   InternetService   5634 non-null   str    
 8   OnlineSecurity    5634 non-null   str    
 9   OnlineBackup      5634 non-null   str    
 10  DeviceProtection  5634 non-null   str    
 11  TechSupport       5634 non-null   str    
 12  StreamingTV       5634 non-null   str    
 13  StreamingMovies   5634 non-null   str    
 14  Contract          5634 non-null   str    
 15  PaperlessBilling  5634 non-null   str    
 16  PaymentMethod     5634 non-null   str    
 17  Monthl

In [16]:
xtrain = train_data.drop("Churn", axis=1)
ytrain = train_data["Churn"]
xtest = test_data.drop("Churn", axis=1)
ytest = test_data["Churn"]

In [17]:
import sys
sys.path.insert(0, str(BASE_DIR / "src"))
from pipeline import build_pipeline
print("Pipeline imported successfully!")

Pipeline imported successfully!


In [18]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [19]:
classifiers = {
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "SVM": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    )
}

In [21]:
models={}
for name , classifier in classifiers.items():
    pipeline = build_pipeline(xtrain,classifier)
    pipeline.fit(xtrain, ytrain)
    models[name] = pipeline
    print(f"{name} trained successfully!")

Decision Tree trained successfully!
Random Forest trained successfully!


/Users/apple/Desktop/timepass/RetailMind/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM trained successfully!


In [22]:
for name, model in models.items():
    print(f"{name}:")
    print("Fitted:", hasattr(model, "classes_"))
    print("Classes:", model.classes_)
    print("-" * 40)

Decision Tree:
Fitted: True
Classes: [0 1]
----------------------------------------
Random Forest:
Fitted: True
Classes: [0 1]
----------------------------------------
SVM:
Fitted: True
Classes: [0 1]
----------------------------------------


In [23]:
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for name, model in models.items():
    filename = name.lower().replace(" ", "_") + ".joblib"
    model_path = MODEL_DIR / filename
    joblib.dump(model, model_path)
    print(f"Saved {name} at {model_path}")

Saved Decision Tree at /Users/apple/Desktop/timepass/RetailMind/models/decision_tree.joblib
Saved Random Forest at /Users/apple/Desktop/timepass/RetailMind/models/random_forest.joblib
Saved SVM at /Users/apple/Desktop/timepass/RetailMind/models/svm.joblib


In [24]:
baseline_model = joblib.load(
    BASE_DIR / "models" / "churn_pipeline.joblib"
)
print("Day 1 Logistic Regression model loaded!")

Day 1 Logistic Regression model loaded!


In [25]:
all_models = {
    "Logistic Regression": baseline_model,
    "Decision Tree": models["Decision Tree"],
    "Random Forest": models["Random Forest"],
    "SVM": models["SVM"]
}

print("Total models:", len(all_models))

Total models: 4


In [29]:
predictions = {}
probabilities = {}

for name, model in all_models.items():
    predictions[name] = model.predict(xtest)
    probabilities[name] = model.predict_proba(xtest)[:, 1]
    print(f"{name}: predictions generated")

Logistic Regression: predictions generated
Decision Tree: predictions generated
Random Forest: predictions generated
SVM: predictions generated


In [31]:
results = []
for name in all_models:
    y_pred = predictions[name]
    y_prob = probabilities[name]
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(ytest, y_pred),
        "Precision": precision_score(ytest, y_pred, zero_division=0),
        "Recall": recall_score(ytest, y_pred, zero_division=0),
        "F1 Score": f1_score(ytest, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(ytest, y_prob)
    }
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.738112,0.504303,0.783422,0.613613,0.841298
1,Decision Tree,0.740241,0.511173,0.489305,0.500000,0.660348
2,Random Forest,0.767211,0.554245,0.628342,0.588972,0.821168
3,SVM,0.743790,0.511504,0.772727,0.615548,0.821065


In [33]:
results_df = results_df.sort_values(
    by="Recall",
    ascending=False
)
results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.738112,0.504303,0.783422,0.613613,0.841298
3,SVM,0.743790,0.511504,0.772727,0.615548,0.821065
2,Random Forest,0.767211,0.554245,0.628342,0.588972,0.821168
1,Decision Tree,0.740241,0.511173,0.489305,0.500000,0.660348


In [39]:
for name in all_models:
    print("=" * 60)
    print(name)
    print("=" * 60)

    print(
        classification_report(
            ytest,predictions[name], target_names=["no_churn","churn"]
        )
    )

Logistic Regression
              precision    recall  f1-score   support

    no_churn       0.90      0.72      0.80      1035
       churn       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

Decision Tree
              precision    recall  f1-score   support

    no_churn       0.82      0.83      0.82      1035
       churn       0.51      0.49      0.50       374

    accuracy                           0.74      1409
   macro avg       0.66      0.66      0.66      1409
weighted avg       0.74      0.74      0.74      1409

Random Forest
              precision    recall  f1-score   support

    no_churn       0.86      0.82      0.84      1035
       churn       0.55      0.63      0.59       374

    accuracy                           0.77      1409
   macro avg       0.71      0.72      0.71      1409
weighted avg       0.78   

In [41]:

for name in all_models:
    print("=" * 25)
    print(name)
    print("=" * 25)

    print(
        confusion_matrix(
            ytest,predictions[name]
        )
    )

Logistic Regression
[[747 288]
 [ 81 293]]
Decision Tree
[[860 175]
 [191 183]]
Random Forest
[[846 189]
 [139 235]]
SVM
[[759 276]
 [ 85 289]]


In [42]:
best_model_name = results_df.iloc[0]["Model"]
best_model = all_models[best_model_name]

In [43]:
import joblib

BEST_MODEL_PATH = BASE_DIR / "models" / "day2_champion.joblib"
joblib.dump(best_model, BEST_MODEL_PATH)
print("Best model saved:", best_model_name)

Best model saved: Logistic Regression
